# Wound classifier: DINOv2 experiments (GPU)

Runs the frozen-encoder probes and the DINOv2 fine-tune on a free Colab GPU, on the same group-safe split as every other result in `experiments/results.csv`.

**Before running:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all.

Full instructions and the rules for reporting numbers are in `COLLAB.md`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then run all again.'

In [ ]:
!git clone -q https://github.com/shubhisingh1510/AI.git repo
%cd repo
!git log --oneline -3

## 1. Get the images

The images are not stored in the repo. These two scripts download them from their public sources and verify checksums. The split files are already in the repo, so everyone trains and tests on exactly the same images.

In [ ]:
!pip install -q pyyaml
!python scripts/fetch_azh_dataset.py --config configs/config.yaml
!python scripts/fetch_medetec_supplement.py --config configs/config.yaml
!find data/raw -type f | wc -l   # should print 891

## 2. Frozen-encoder probes (about 5 minutes on a GPU)

No training of the encoder: features are extracted once, then a linear classifier is fitted. `dinov2_vitl14` is too slow on a laptop CPU, which is why it is run here.

In [ ]:
!python src/foundation_probe.py --encoders resnet50 dinov2_vits14 dinov2_vitb14 dinov2_vitl14 2>&1 | grep -E 'test acc|AZH subset|==='

## 3. Fine-tune DINOv2 ViT-B (main split, then 5-fold CV)

Roughly 20 to 30 minutes for the main split and about two hours with `--cv` on a T4. If Colab disconnects, run this cell again; nothing else needs re-running.

Things worth trying, one change at a time: `--size 224` vs `448`, `--lr 2e-5` / `1e-4`, `--layer-decay 0.65` / `0.85`, `--model dinov2_vitl14 --batch-size 8`.

In [ ]:
!python src/finetune_dinov2.py --model dinov2_vitb14 --size 448 --epochs 30 --cv

## 4. Save what you got

Download the result files and add them to the repo (see `COLLAB.md`, "Recording a result"). Report the number you got, including when it is lower than a previous one.

In [ ]:
!ls results/foundation_probe_metrics.json results/finetune_*_metrics.json
!zip -q dinov2_results.zip results/foundation_probe_metrics.json results/finetune_*_metrics.json
from google.colab import files
files.download('dinov2_results.zip')